# Master Inference Notebook

This notebook runs room-wise LoRA inference for multiple rooms in one place.
It preserves your Google Drive paths and lets you scale to many layouts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import getpass
import os

# Optional: set HF token if needed
# hf_token = getpass.getpass('Enter your HF access token and press enter: ')
# os.environ['HF_TOKEN'] = hf_token
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print('HF transfer enabled')

In [ ]:
!pip install -q hf_transfer
!pip install -q -U controlnet-aux
!pip install -q diffusers accelerate safetensors

In [ ]:
import glob
import torch
from PIL import Image
from diffusers import FluxContxrolPipeline
from diffusers.utils import load_image
from controlnet_aux import CannyDetector

In [ ]:
# ==== DRIVE PATHS (PRESERVED) ====
DRIVE_ROOT = '/content/drive/MyDrive'
MASK_ROOT = f'{DRIVE_ROOT}/extraction'
OUTPUT_ROOT = f'{DRIVE_ROOT}/generated'

ROOM_CONFIGS = {
    'living_room': {
        'mask_subdir': 'living_room',
        'mask_pattern': 'living_room.png',
        'lora_path': '/content/drive/MyDrive/outputs_livingroom_new/my_second_flux_lora_v2_living/my_second_flux_lora_v2_living.safetensors',
        'adapter_name': 'living_room',
        'output_subdir': 'living_room',
        'prompt': 'top-down 2d floor plan of a modern living room, colored, clean lines, NO FURNITURE HALLUCINATIONS',
        'negative_prompt': 'bad architecture, overlapping furniture, furniture outside room'
    },
    'bedroom': {
        'mask_subdir': 'bedroom',
        'mask_pattern': 'bedroom.png',
        'lora_path': '/content/drive/MyDrive/outputs_bedroom_modified/my_first_flux_lora_v2_bedroom/my_first_flux_lora_v2_bedroom_000003500.safetensors',
        'adapter_name': 'bedroom',
        'output_subdir': 'bedroom',
        'prompt': 'top-down 2d floor plan of a modern bedroom, colored, clean lines, NO FURNITURE HALLUCINATIONS',
        'negative_prompt': 'bad architecture, overlapping furniture, furniture outside room'
    },
    'bathroom': {
        'mask_subdir': 'bathroom',
        'mask_pattern': 'bathroom.png',
        'lora_path': '/content/drive/MyDrive/outputs_bathroom_modified/my_first_flux_lora_v2_bathroom/my_first_flux_lora_v2_bathroom_000003500.safetensors',
        'adapter_name': 'bathroom',
        'output_subdir': 'bathroom',
        'prompt': 'top-down 2d floor plan of a modern bathroom, colored, clean lines, NO FURNITURE HALLUCINATIONS',
        'negative_prompt': 'bad architecture, overlapping furniture, furniture outside room'
    },
    'hallway': {
        'mask_subdir': 'passageway',
        'mask_pattern': 'passageway.png',
        'lora_path': '/content/drive/MyDrive/outputs_hallways/my_first_flux_lora_v2_hallway/my_first_flux_lora_v2_hallway.safetensors',
        'adapter_name': 'hallway',
        'output_subdir': 'hallway',
        'prompt': 'top-down 2d floor plan of a modern hallway, colored, clean lines, NO FURNITURE HALLUCINATIONS',
        'negative_prompt': 'bad architecture, overlapping furniture, furniture outside room'
    },
    'toilet': {
        'mask_subdir': 'toilet',
        'mask_pattern': 'toilet.png',
        'lora_path': '/content/drive/MyDrive/outputs_toilet/my_first_flux_lora_v2_toilet/my_first_flux_lora_v2_toilet.safetensors',
        'adapter_name': 'toilet',
        'output_subdir': 'toilet',
        'prompt': 'top-down 2d floor plan of a modern toilet, colored, clean lines, NO FURNITURE HALLUCINATIONS',
        'negative_prompt': 'bad architecture, overlapping furniture, furniture outside room'
    },
}

print('Configured rooms:', list(ROOM_CONFIGS.keys()))

In [ ]:
# Initialize pipeline + canny once
canny_processor = CannyDetector()

pipe = FluxControlPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-dev',
    torch_dtype=torch.float16
)
pipe.to('cuda')

# Load the Canny ControlNet LoRA once
pipe.load_lora_weights(
    'black-forest-labs/FLUX.1-Canny-dev-lora',
    adapter_name='canny'
)

# Optional memory optimization
pipe.enable_model_cpu_offload()

In [ ]:
def run_room_inference(room_key, cfg, num_variations=1, steps=30, guidance=7.5):
    room_mask_dir = os.path.join(MASK_ROOT, cfg['mask_subdir'])
    output_dir = os.path.join(OUTPUT_ROOT, cfg['output_subdir'])
    os.makedirs(output_dir, exist_ok=True)

    mask_files = glob.glob(os.path.join(room_mask_dir, cfg['mask_pattern']))
    if not mask_files:
        print(f'No masks found for {room_key} in {room_mask_dir}')
        return

    # Load room LoRA once
    pipe.load_lora_weights(cfg['lora_path'], adapter_name=cfg['adapter_name'])
    pipe.set_adapters([cfg['adapter_name'], 'canny'], adapter_weights=[0.8, 0.7])

    for mask_path in mask_files:
        mask_image = load_image(mask_path)
        control_image = canny_processor(
            mask_image,
            low_threshold=50,
            high_threshold=200,
            detect_resolution=512,
            image_resolution=512
        )

        for i in range(num_variations):
            result = pipe(
                prompt=cfg['prompt'],
                negative_prompt=cfg['negative_prompt'],
                control_image=control_image,
                num_inference_steps=steps,
                guidance_scale=guidance
            )
            image = result.images[0]
            base = os.path.splitext(os.path.basename(mask_path))[0]
            out_name = f'{room_key}_{base}_var_{i+1}.png'
            out_path = os.path.join(output_dir, out_name)
            image.save(out_path)
            print(f'Saved {out_path}')

In [ ]:
# Run all rooms
# Adjust num_variations per room to scale output
for room_key, cfg in ROOM_CONFIGS.items():
    print(f'
=== Running {room_key} ===')
    run_room_inference(room_key, cfg, num_variations=3, steps=30, guidance=7.5)